## Imports

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import os

ROOT_CODE_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils").exists())

if str(ROOT_CODE_DIR) not in sys.path:
    sys.path.append(str(ROOT_CODE_DIR))

from utils.inference_utils import EVALUATION_MAIN_ASPECTS, EVALUATION_FIVE_ASPECTS
from utils.data_setup import get_dataset_file_path

## Config and Constants

In [ ]:
NUM_TRIALS = 1

## Data Loading

In [ ]:
# Both stages schedule the same tasks; the ratings paths also give the few-shot samples file.
uicrit_file, base64_screens_file, few_shot_samples_file = get_dataset_file_path("ratings")

uicrit_df = pd.read_parquet(uicrit_file)
screenshots_df = pd.read_parquet(base64_screens_file)

if few_shot_samples_file:
    few_shot_samples_df = pd.read_parquet(few_shot_samples_file)
else:
    few_shot_samples_df = None

print("UICrit rows:", len(uicrit_df))
print("Screens rows:", len(screenshots_df))

In [ ]:
schedule_dir = ROOT_CODE_DIR / "experiment_notebooks" / "batch_schedules"
schedule_dir.mkdir(exist_ok=True)

rating_schedule_file_1000_tasks = schedule_dir / "rating_trial_schedule-1000_tasks.parquet"
rating_schedule_file_all_tasks = schedule_dir / "rating_trial_schedule-all_tasks.parquet"
critiques_schedule_file_1000_tasks = schedule_dir / "critiques_trial_schedule-1000_tasks.parquet"
critiques_schedule_file_all_tasks = schedule_dir / "critiques_trial_schedule-all_tasks.parquet"
critiques_schedule_file_all_tasks_excluding_1000 = schedule_dir / "critiques_trial_schedule-all_tasks_excluding_1000.parquet"

## Preparing Schedules

Use these cells only when you need to build or refresh the schedule parquet files in `batch_schedules/`.

### Ratings Schedule

In [ ]:
# Build ratings schedule for all tasks
records = []
for _, row in uicrit_df.iterrows():
    screen_task_id = row["screen_task_id"]
    for trial in range(1, NUM_TRIALS + 1):
        for aspect in EVALUATION_MAIN_ASPECTS:
            record = {
                "custom_id": (
                    f"{screen_task_id}&trial{trial}&aspect={aspect}"
                    if NUM_TRIALS > 1
                    else f"{screen_task_id}&aspect={aspect}"
                ),
                "screen_task_id": screen_task_id,
                "aspect": aspect,
            }
            if NUM_TRIALS > 1:
                record["trial"] = trial
            records.append(record)

trials_schedule_df = pd.DataFrame.from_records(records)
trials_schedule_df.to_parquet(rating_schedule_file_all_tasks, index=False)

print(trials_schedule_df.shape)
trials_schedule_df.head(4)

In [ ]:
# Build ratings schedule for the first task of 1000 screens
trials_schedule_df = pd.read_parquet(rating_schedule_file_all_tasks)
first_tasks_df = uicrit_df.drop_duplicates(subset="screen_id", keep="first")
selected_task_ids = first_tasks_df["screen_task_id"].head(1000).unique()

trials_schedule_df = trials_schedule_df[trials_schedule_df["screen_task_id"].isin(selected_task_ids)].copy()
trials_schedule_df.to_parquet(rating_schedule_file_1000_tasks, index=False)

print(trials_schedule_df.shape)
trials_schedule_df.head(4)

In [ ]:
# Optional: rebuild few-shot samples file for ratings
uicrit_df_ratings = uicrit_df.copy()
uicrit_df_ratings["total_rating"] = uicrit_df_ratings[EVALUATION_FIVE_ASPECTS].sum(axis=1)
uicrit_df_ratings = uicrit_df_ratings.sort_values(by="total_rating")

lowest_rated = uicrit_df_ratings.head(3).drop(columns=["total_rating", "comments_source", "comments"], errors="ignore")
mid_rated = uicrit_df_ratings.iloc[[len(uicrit_df_ratings) // 2]].drop(columns=["total_rating", "comments_source", "comments"], errors="ignore")
highest_rated = uicrit_df_ratings.tail(3).drop(columns=["total_rating", "comments_source", "comments"], errors="ignore")

few_shot_samples_df = pd.concat([lowest_rated, mid_rated, highest_rated]).merge(screenshots_df, on="screen_id", how="left")
few_shot_samples_df.to_parquet(few_shot_samples_file, index=False)

few_shot_samples_df.head()

### Critiques Schedule

In [ ]:
# Build critiques schedule for all tasks
records = []
for _, row in uicrit_df.iterrows():
    screen_task_id = row["screen_task_id"]
    records.append({
        "custom_id": screen_task_id,
        "screen_task_id": screen_task_id,
    })

critiques_schedule_df = pd.DataFrame.from_records(records)
critiques_schedule_df.to_parquet(critiques_schedule_file_all_tasks, index=False)

print(critiques_schedule_df.shape)
critiques_schedule_df.head(4)

In [ ]:
# Build critiques schedule for the first task of 1000 screens
critiques_schedule_df = pd.read_parquet(critiques_schedule_file_all_tasks)
first_tasks_df = uicrit_df.drop_duplicates(subset="screen_id", keep="first")
selected_task_ids = first_tasks_df["screen_task_id"].head(1000).unique()

critiques_schedule_df = critiques_schedule_df[critiques_schedule_df["screen_task_id"].isin(selected_task_ids)].copy()
critiques_schedule_df.to_parquet(critiques_schedule_file_1000_tasks, index=False)

print(critiques_schedule_df.shape)
critiques_schedule_df.head(4)

In [ ]:
# Optional: build critiques schedule for all_tasks excluding the 1000-task subset
schedule_1000_tasks_df = pd.read_parquet(critiques_schedule_file_1000_tasks)
all_tasks_schedule_df = pd.read_parquet(critiques_schedule_file_all_tasks)

schedule_all_tasks_excluding_1000_df = all_tasks_schedule_df[
    ~all_tasks_schedule_df["screen_task_id"].isin(schedule_1000_tasks_df["screen_task_id"])
].copy()

schedule_all_tasks_excluding_1000_df.to_parquet(critiques_schedule_file_all_tasks_excluding_1000, index=False)

print(schedule_all_tasks_excluding_1000_df.shape)
schedule_all_tasks_excluding_1000_df.head(4)